# SLM 相位优化工作流程

## 概述
本 Notebook 用于生成和优化空间光调制器 (SLM) 的相位图案，支持微透镜阵列的设计与优化。

## 工作流程
1. **生成任务列表** - 通过 GUI 界面配置优化参数，创建批量任务
2. **运行优化任务** - 批量执行相位优化，保存结果到 `./output/`
3. **查看结果** - 浏览、可视化优化结果，可直接上传到 SLM

## 输出文件说明
每个任务会在 `./output/{job_title}/` 目录下生成：
- `{job_title}.npy` - 8-bit 相位图 (可直接上传到 SLM)
- `{job_title}.json` - 优化参数记录
- `{job_title}_optimizer.pkl` - 完整优化器对象 (用于后续可视化)

---

# Step 1: 生成任务列表

In [2]:
"""
Step 1: 加载配置并启动 GUI
=============================
功能：
- 从 JSON 配置文件加载默认光学参数
- 启动交互式 GUI 界面，用于配置和管理优化任务

GUI 使用说明：
- 左侧面板：设置光学参数 (M, 焦距, 重叠比例等)
- 右侧面板：管理任务列表
- 点击 "Add to Job List" 将当前配置添加到任务队列
- 支持批量添加多个不同参数的任务
"""

from optics_utils import load_dict_from_json
from phase_optimizer_gui import create_optimizer_gui
import os

# ============================================================
# 配置文件选择
# ============================================================
# 基础配置文件（包含默认光学参数）
filename_base = r"base.json"
# 也可以加载之前保存的优化配置：
# filename_base = r"251105_1_super_0.9.json"

# 配置文件目录
path_json = r".\\config\\"

# ============================================================
# 加载配置并启动 GUI
# ============================================================
# 加载 JSON 配置文件
params = load_dict_from_json(os.path.join(path_json, filename_base))

# 创建优化器 GUI
# 参数说明：
#   - default_params: 预加载的默认参数
# GUI 中可配置的主要参数：
#   - M: 微透镜阵列大小 (如 5 表示 5x5 阵列)
#   - focal_length: 焦距 (mm)
#   - overlap_ratio: 相邻透镜重叠比例 (0~1)
#   - airy_correction: Airy 斑校正因子
#   - depth_in_focus: 景深范围 (DOF 单位)
#   - mode: 'fresnel' (直接生成) 或 'optimized' (优化生成)
gui = create_optimizer_gui(default_params=params)

# Step 2: 运行优化任务

执行 GUI 中添加的所有任务。支持三种运行模式：
- **仿真模式** (`sim_mode=True`): 仅进行优化计算，不连接实际硬件
- **远程模式** (`remote_mode=True`): 通过 RPyC 连接远程 SLM
- **本地模式**: 直接连接本地 SLM

> **注意**: 优化过程需要 GPU 加速，请确保 CUDA 可用

In [ ]:
"""
Step 2: 批量执行优化任务
=========================
功能：
- 遍历 GUI 中的任务列表，逐个执行优化
- 保存优化结果到 ./output/ 目录
- 支持 Fresnel 模式（直接生成）和 Optimized 模式（梯度优化）

输出：
- 每个任务生成独立文件夹，包含 .npy, .json, .pkl 文件
"""

from batch_processor import process_jobs
from hardware import RemoteSLMManager, SLMManager
import torch

# ============================================================
# 设备选择：优先使用 GPU (CUDA)
# ============================================================
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"使用设备: {device}")

# ============================================================
# SLM 连接模式选择
# ============================================================
# 模式说明：
#   sim_mode=True  : 仿真模式，不连接任何硬件（推荐用于纯优化）
#   remote_mode=True: 远程模式，通过 RPyC 连接远程 SLM 服务
#   两者都为 False : 本地模式，直接连接本地 SLM 硬件

remote_mode = True   # 是否使用远程 SLM
sim_mode = True      # 是否使用仿真模式（不连接硬件）

# 根据模式选择创建 SLM 管理器
if sim_mode:
    # 仿真模式：不连接实际 SLM，仅进行优化计算
    slm_manager = SLMManager(sim_mode=True)
elif remote_mode:
    # 远程模式：连接远程 SLM 服务（需要先启动 RPyC 服务端）
    slm_manager = RemoteSLMManager()
else: 
    # 本地模式：直接连接本地 SLM 硬件
    slm_manager = SLMManager(sim_mode=False)

# ============================================================
# 执行批量优化
# ============================================================
# process_jobs 参数说明：
#   gui: GUI 实例，包含任务列表
#   slm_manager: SLM 管理器实例
#   device: 计算设备 (cuda/cpu)
#   output_dir: 输出目录 (默认 './output')
#   save_optimizer: 是否保存优化器对象 (默认 True)
#   upsampling: 优化时的上采样因子 (默认 2.0)

results = process_jobs(gui, slm_manager=slm_manager, device=device)

# results 是一个字典，键为任务名称，值包含：
#   - 'status': 'success' 或 'error'
#   - 'output_dir': 输出目录路径
#   - 'npy_path': 相位图文件路径
#   - 'json_path': 参数文件路径
#   - 'optimizer_path': 优化器对象路径
#   - 'optimizer': 优化器实例（可用于进一步分析）

# Step 3: 查看和管理结果

使用交互式浏览器查看已保存的优化结果：
- **选择任务**: 点击列表中的任务查看详情
- **可视化**: 点击 "Visualize" 按钮生成可视化图表
- **上传 SLM**: 如果连接了 SLM，可直接上传相位图

可视化内容包括：
- 相位图分布
- 目标 PSF vs 实际 PSF 对比
- 能量分布和聚焦效率

In [ ]:
"""
Step 3: 浏览和可视化优化结果
=============================
功能：
- 扫描 ./output/ 目录下的所有已保存任务
- 提供交互式 GUI 浏览和选择任务
- 加载优化器对象并生成可视化图表
- 支持直接上传相位图到 SLM

GUI 说明：
- ✓ 表示该任务包含完整的优化器对象 (.pkl)
- ○ 表示该任务仅有相位图和参数文件
"""

from batch_processor import browse_jobs

# ============================================================
# 创建结果浏览器
# ============================================================
# browse_jobs 参数说明：
#   output_dir: 搜索任务的目录 (默认 './output')
#   upsampling: 可视化时的上采样因子 (默认 3)
#   slm_manager: 可选，如果提供则支持直接上传到 SLM

browser = browse_jobs(output_dir='./output')

# ============================================================
# 获取当前选中的优化器（可选）
# ============================================================
# 在 GUI 中选择并可视化任务后，可通过以下方式获取优化器对象：
# optimizer = browser.get_current_optimizer()

# 获取当前选中的相位图：
# phase_8bit = browser.get_current_phase()

# 获取任务列表：
# job_list = browser.get_job_list()